In [1]:
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}, GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.10.1, GPU: True


In [5]:
sentences = [
    "i love machine learning",
    "machine learning is fun",
    "i love deep learning",
    "deep learning is powerful",
    "i enjoy learning new things",
    "natural language processing is fascinating",
    "neural networks can recognize patterns",
    "computer vision helps machines see",
    "reinforcement learning trains agents through rewards",
    "transformers revolutionized natural language processing",
    "attention mechanisms allow models to focus on important information",
    "large language models generate human like text",
    "convolutional neural networks excel at image classification",
    "recurrent neural networks are good for sequential data",
    "lstm networks remember information over long periods",
    "generative adversarial networks create realistic synthetic data",
    "autoencoders learn efficient data representations",
    "transfer learning reduces the need for large datasets",
    "data preprocessing is crucial for model performance",
    "feature engineering transforms raw data into useful inputs",
    "hyperparameter tuning optimizes model behavior",
    "cross validation helps prevent overfitting",
    "regularization techniques improve model generalization",
    "dropout prevents neural networks from overfitting",
    "batch normalization accelerates deep network training",
    "gradient descent minimizes the loss function",
    "backpropagation computes gradients for neural networks",
    "activation functions introduce non linearity",
    "relu is a popular activation function",
    "sigmoid outputs values between zero and one",
    "softmax converts logits to probability distributions",
    "loss functions measure prediction errors",
    "accuracy is a common evaluation metric",
    "precision and recall matter for imbalanced datasets",
    "f1 score balances precision and recall",
    "roc curves show classifier performance",
    "confusion matrices visualize classification results",
    "decision trees split data based on features",
    "random forests combine multiple decision trees",
    "gradient boosting builds trees sequentially",
    "support vector machines find optimal decision boundaries",
    "k nearest neighbors uses similarity for classification",
    "k means clustering groups similar data points",
    "principal component analysis reduces data dimensions",
    "t sne visualizes high dimensional data",
    "pca preserves variance while reducing features",
    "linear regression models relationships between variables",
    "logistic regression predicts binary outcomes",
    "naive bayes assumes feature independence",
    "ensemble methods combine multiple models for better performance",
    "stacking trains a meta model on base predictions",
    "bagging reduces variance by averaging models",
    "boosting focuses on hard to classify examples",
    "xception is a depthwise separable convolutional architecture",
    "resnet uses skip connections to train deeper networks",
    "inception modules capture multi scale features",
    "mobilenet is optimized for mobile devices",
    "efficientnet achieves state of the art efficiency",
    "bert understands context in both directions",
    "gpt generates coherent and contextual text",
    "t5 treats all nlp tasks as text to text problems",
    "clip connects images and text in a shared embedding space",
    "dalle generates images from text descriptions",
    "stable diffusion creates high quality images iteratively",
    "whisper transcribes speech accurately",
    "yolo detects objects in real time",
    "ssd performs fast single shot object detection",
    "mask r cnn segments objects at pixel level",
    "unet is effective for biomedical image segmentation",
    "alpha go mastered the game of go using reinforcement learning",
    "alphafold predicts protein structures accurately",
    "self supervised learning learns from unlabeled data",
    "contrastive learning pulls similar samples together",
    "simclr is a simple framework for contrastive learning",
    "few shot learning adapts to new tasks with few examples",
    "zero shot learning generalizes to unseen classes",
    "meta learning learns how to learn efficiently",
    "multimodal learning combines text images and audio",
    "federated learning trains models without centralizing data",
    "explainable ai makes model decisions interpretable",
    "shap values explain feature importance for predictions",
    "lime provides local explanations for any classifier",
    "attention maps visualize what models focus on",
    "feature visualization shows what neurons detect",
    "adversarial examples fool machine learning models",
    "adversarial training improves model robustness",
    "differential privacy protects individual data points",
    "fairness in ai prevents algorithmic bias",
    "responsible ai ensures ethical machine learning practices",
    "auto ml automates the machine learning workflow",
    "keras provides a user friendly deep learning api",
    "tensorflow is a powerful numerical computation library",
    "pytorch offers dynamic computation graphs",
    "scikit learn provides classical ml algorithms",
    "pandas simplifies data manipulation and analysis",
    "numpy enables efficient array operations",
    "matplotlib creates static visualizations",
    "seaborn makes statistical plots beautiful",
    "jupyter notebooks support interactive development",
    "google colab offers free gpu computing",
    "hugging face hosts pretrained transformer models",
    "opencv processes images and videos in real time",
    "nltk provides tools for working with human language data",
    "spacy is industrial strength natural language processing",
    "gensim implements topic modeling and word embeddings",
    "word2vec learns vector representations of words",
    "glove captures global word co occurrence statistics",
    "fasttext handles out of vocabulary words well",
    "bert embeddings are contextually aware"
]

In [30]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

VOCAB_SIZE = len(tokenizer.word_index) + 1
print(f"vocab size is : {VOCAB_SIZE}")

sequences = []
for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]  # Input:  ["sentence"]  - > Output: [[tokens]]
    sequences.append(token_list)

MAX_LEN_SEQ = max(len(seq) for seq in sequences)
sequences = pad_sequences(sequences, maxlen=MAX_LEN_SEQ, padding='post')
print(f"total sequences are {len(sequences)} and max length of sequence is : {MAX_LEN_SEQ}  ")

# GPT-style: input is all but last, target is shifted by 1
X = sequences[:, :-1]  # [num_sentences, max_len-1]
y = sequences[:, 1:]  # [num_sentences, max_len-1]  ← SHIFTED, not just last token
# y = sequences[:, -1]          # n-gram style

# PADDING MASK — 1 where real token, 0 where pad
padding_mask = tf.cast(y != 0, tf.float32)  # shape: [num_sentences, max_len-1]

print(f"X shape: {X.shape}")  # shape: [num_sentences, max_len]
print(f"y shape: {y.shape}")
print(f"mask shape: {padding_mask.shape}")

vocab size is : 448
total sequences are 109 and max length of sequence is : 10  
X shape: (109, 9)
y shape: (109, 9)
mask shape: (109, 9)


In [31]:
d_model = 64
num_heads = 4

In [32]:
def create_causal_mask(seq_len):
    """Lower triangular matrix ( mask)"""
    mask = tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0)
    return mask


class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()

        # Masked self-attention
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads,
                                                      key_dim=d_model // num_heads)

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(2 * d_model, activation='gelu'),
            tf.keras.layers.Dropout(0.1),
            tf.keras.layers.Dense(d_model),
            tf.keras.layers.Dropout(0.1),
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout = tf.keras.layers.Dropout(0.1)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]  #num of features=9
        mask = create_causal_mask(seq_len)  # [seq_len, seq_len] bool

        attn = self.mha(query=x, key=x, value=x, attention_mask=mask,training=training)
        attn = self.dropout(attn, training=training)
        x = self.norm1(x + attn)  # [batch, seq_len, d_model]

        ffn_out = self.ffn(x, training=training)
        return self.norm2(x + ffn_out)  # [batch, seq_len, d_model]


class GPTMini(tf.keras.Model):
    def __init__(self, vocab_size=VOCAB_SIZE, max_len=MAX_LEN_SEQ - 1, d_model=d_model, num_heads=num_heads):
        super().__init__()
        self.token_embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_embedding = tf.keras.layers.Embedding(max_len, d_model)
        self.decoder = DecoderBlock(d_model, num_heads)
        self.norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dense = tf.keras.layers.Dense(vocab_size)  # no activation — from_logits=True

    def call(self, x, training=False):
        # print("Input:", x.shape)  #   input (2,9)
        seq_len = tf.shape(x)[1]  # (109, 9)
        positions = tf.range(start=0, limit=seq_len, delta=1)  # (9,)
        x = self.token_embedding(x) + self.pos_embedding(positions)  # [batch, seq_len, d_model]
        # print("Token Embedding:", tok_emb.shape)        # Token Embedding: (2, 9, 64)
        # print("Pos Embedding:", pos_emb.shape)          # Pos Embedding: (9, 64)
        x = self.decoder(x, training=training)  # [batch, seq_len, d_model]
        # print("After Decoder:", x.shape)        # After Decoder: (2, 9, 64)
        x = self.norm(x)  # [batch, seq_len, d_model]
        return self.dense(x)  # [batch, seq_len, vocab_size]
        # print("Final Output:", x.shape)     # Final Output: (2, 9, 448)
        # NO x[:, -1, :] — we keep all positions for GPT-style training


model = GPTMini(VOCAB_SIZE, MAX_LEN_SEQ - 1)  # max_len-1 because X = sequences[:, :-1]

In [33]:
dummy = tf.zeros((2, MAX_LEN_SEQ - 1), dtype=tf.int32)
out = model(dummy)
print(f"Input  shape: {dummy.shape}")  # (2, max_len-1)
print(f"Output shape: {out.shape}")  # (2, max_len-1, vocab_size)

Input  shape: (2, 9)
Output shape: (2, 9, 448)


In [34]:
model.compile(optimizer=tf.keras.optimizers.Adam(3e-4),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

model.fit(X, y, sample_weight=padding_mask,
          epochs=10, batch_size=32, verbose=1)

Epoch 1/10
4/4 [==============================] - 1s 26ms/step - loss: 3.7008 - accuracy: 0.0041
Epoch 2/10
4/4 [==============================] - 0s 36ms/step - loss: 3.6325 - accuracy: 0.0051
Epoch 3/10
4/4 [==============================] - 0s 27ms/step - loss: 3.5730 - accuracy: 0.0082
Epoch 4/10
4/4 [==============================] - 0s 34ms/step - loss: 3.5283 - accuracy: 0.0133
Epoch 5/10
4/4 [==============================] - 0s 28ms/step - loss: 3.4735 - accuracy: 0.0133
Epoch 6/10
4/4 [==============================] - 0s 26ms/step - loss: 3.4204 - accuracy: 0.0194
Epoch 7/10
4/4 [==============================] - 0s 33ms/step - loss: 3.3771 - accuracy: 0.0265
Epoch 8/10
4/4 [==============================] - 0s 29ms/step - loss: 3.3382 - accuracy: 0.0296
Epoch 9/10
4/4 [==============================] - 0s 27ms/step - loss: 3.2887 - accuracy: 0.0408
Epoch 10/10
4/4 [==============================] - 0s 25ms/step - loss: 3.2455 - accuracy: 0.0581


In [35]:
loss_no_mask = model.evaluate(X[:10], y[:10], verbose=0)
loss_with_mask = model.evaluate(X[:10], y[:10], sample_weight=padding_mask[:10], verbose=0)

print(f"Loss without mask: {loss_no_mask[0]:.4f}")
print(f"Loss with mask:    {loss_with_mask[0]:.4f}")
# with mask should be HIGHER — it's only averaging over real tokens

Loss without mask: 5.6875
Loss with mask:    2.1665


In [37]:
def generate_text(model, tokenizer, seed_text, num_words=10, max_len=MAX_LEN_SEQ):
    reverse_word_index = {v: k for k, v in tokenizer.word_index.items()}
    result = seed_text

    for _ in range(num_words):
        # Tokenize current text
        token_list = tokenizer.texts_to_sequences([result])[0]
        input_len = len(token_list)

        # Stop if we've exceeded max length
        if input_len >= max_len - 1:
            break

        # Pad
        padded = pad_sequences([token_list], maxlen=max_len - 1, padding='post')

        # Predict
        logits = model.predict(padded, verbose=0)  # [1, seq_len, vocab_size]
        next_logits = logits[0, input_len - 1, :]  # last real position

        # Get top prediction
        predicted_id = tf.argmax(next_logits).numpy()

        # Stop if pad token predicted
        if predicted_id == 0:
            break

        predicted_word = reverse_word_index.get(predicted_id, '<UNK>')
        result += ' ' + predicted_word

    return result


# Test
print(generate_text(model, tokenizer, seed_text="computer vision", num_words=10))
print(generate_text(model, tokenizer, seed_text="computer vision helps", num_words=8))

computer vision numerical by for for for makes trees
computer vision helps for for for for makes trees
